In [ ]:
import os
import unicodedata
import zipfile

import numpy as np
import pandas as pd

try:
    from google.colab import files as colab_files
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

CARPETA = "modelo_limpio"
ZIP_SALIDA = "modelo_limpio.zip"

# ------------------------------------------------------------------ 0. ARCHIVO DE ENTRADA
if EN_COLAB:
    print("Sube el archivo 'Hospital_360_dataset_6000reg.xlsx'")
    RUTA_EXCEL = next(iter(colab_files.upload()))
else:
    RUTA_EXCEL = os.environ.get("RUTA_EXCEL", "Hospital_360_dataset_6000reg.xlsx")


def norm(s):
    """minusculas sin tildes ni espacios sobrantes (para que 'Giron' == 'Girón')."""
    return "".join(c for c in unicodedata.normalize("NFD", str(s).strip().lower()) if unicodedata.category(c) != "Mn")


# ------------------------------------------------------------------ 1. CARGA
print("Cargando datos...")
x = pd.read_excel(RUTA_EXCEL, sheet_name=None)
f = x["fact_atenciones"].copy()
dim_pac, dim_hos, dim_ser = x["dim_pacientes"].copy(), x["dim_hospitales"].copy(), x["dim_servicios"].copy()
dim_dia, dim_mun, dim_ase = x["dim_diagnosticos"].copy(), x["dim_municipios"].copy(), x["dim_aseguradoras"].copy()

# ------------------------------------------------------------------ 2. VALIDACION (queda como evidencia en la bitacora)
bitacora = []


def chequeo(nombre, ok, detalle=""):
    bitacora.append({"chequeo": nombre, "resultado": "OK" if ok else "REVISAR", "detalle": str(detalle)})


chequeo("atencion_id unico (PK de la fact)", f.atencion_id.is_unique, f"{f.atencion_id.duplicated().sum()} repetidos")
chequeo("sin filas 100% duplicadas", not f.duplicated().any(), f"{f.duplicated().sum()} duplicadas")
for col, dim, key in [("paciente_id", dim_pac, "paciente_id"), ("hospital_id", dim_hos, "hospital_id"),
                      ("servicio_id", dim_ser, "servicio_id"), ("diagnostico_id", dim_dia, "diagnostico_id")]:
    chequeo(f"PK unica en dim de {key}", dim[key].is_unique)
    chequeo(f"FK {col} sin huerfanos", f[col].isin(dim[key]).all(), f"{(~f[col].isin(dim[key])).sum()} huerfanos")
chequeo("triage entre 1 y 5", f.triage.between(1, 5).all())
chequeo("satisfaccion entre 1 y 5 (sin contar nulos)", f.satisfaccion_1_a_5.dropna().between(1, 5).all())
for c in ["tiempo_espera_min", "costo_atencion_cop", "satisfaccion_1_a_5"]:
    chequeo(f"nulos en {c} (se conservan como NULL)", True, f"{f[c].isna().sum()} ({f[c].isna().mean():.2%})")
chequeo("pacientes con atenciones vs en dim_pacientes", True, f"{f.paciente_id.nunique()} de {len(dim_pac)}")

# ------------------------------------------------------------------ 3. LIMPIEZA DE LA FACT
print("Limpiando tabla de hechos...")
f["fecha_hora_ingreso"] = pd.to_datetime(f["fecha_hora_ingreso"])
f["fecha"] = f["fecha_hora_ingreso"].dt.normalize()  # se deriva de la fecha-hora: garantiza coherencia

# Banderas de calidad (los nulos NO se borran ni se rellenan)
f["espera_faltante"] = f["tiempo_espera_min"].isna().astype(int)
f["costo_faltante"] = f["costo_atencion_cop"].isna().astype(int)
f["satisfaccion_faltante"] = f["satisfaccion_1_a_5"].isna().astype(int)
f["espera_cero"] = (f["tiempo_espera_min"] == 0).astype(int)
f["espera_mayor_120"] = np.where(f["tiempo_espera_min"].isna(), np.nan, (f["tiempo_espera_min"] > 120).astype(float))
f["satisfaccion_baja"] = np.where(f["satisfaccion_1_a_5"].isna(), np.nan, (f["satisfaccion_1_a_5"] <= 3).astype(float))
f["remision_hospitalizada"] = ((f["resultado_atencion"] == "Remision") & (f["es_hospitalizado"] == 1)).astype(int)

# Historial por paciente
f = f.sort_values(["paciente_id", "fecha_hora_ingreso"]).reset_index(drop=True)
f["n_atencion_paciente"] = f.groupby("paciente_id").cumcount() + 1
f["dias_desde_atencion_previa"] = f.groupby("paciente_id")["fecha_hora_ingreso"].diff().dt.total_seconds() / 86400
f["reingreso_sin_antecedente_30d"] = ((f["reingreso_30_dias"] == 1) & ~(f["dias_desde_atencion_previa"] <= 30)).astype(int)
chequeo("reingresos=1 sin atencion previa <=30 d en el dataset (limitacion)", True,
        f"{f.reingreso_sin_antecedente_30d.sum()} de {f.reingreso_30_dias.sum()}")

# Variables para graficar
f["hora"] = f["fecha_hora_ingreso"].dt.hour
f["franja_horaria"] = pd.cut(f["hora"], [-1, 5, 11, 17, 23],
                             labels=["Madrugada (0-5)", "Manana (6-11)", "Tarde (12-17)", "Noche (18-23)"]).astype(str)
labs = ["0-30", "31-60", "61-90", "91-120", "121-180", ">180"]
f["rango_espera"] = pd.cut(f["tiempo_espera_min"], [-1, 30, 60, 90, 120, 180, 10_000], labels=labs).astype(object).fillna("Sin dato")
f["rango_espera_orden"] = f["rango_espera"].map({l: i + 1 for i, l in enumerate(labs)}).fillna(7).astype(int)
f = f.drop(columns=["mes", "dia_semana"])  # redundantes: los aporta dim_tiempo

# ------------------------------------------------------------------ 4. DIMENSIONES
print("Construyendo dimensiones...")
# 4.1 Hospitales: primero se crea municipio_id (el Excel solo trae el nombre) y DESPUES se usa en dim_municipios
mun_map = {norm(n): i for n, i in zip(dim_mun.municipio, dim_mun.municipio_id)}
dim_hos["municipio_id"] = dim_hos["municipio"].map(lambda s: mun_map.get(norm(s)))
chequeo("todos los hospitales con municipio_id tras homologar (Giron/Girón)", dim_hos.municipio_id.notna().all(),
        dim_hos.loc[dim_hos.municipio_id.isna(), "municipio"].tolist())
assert dim_hos.municipio_id.notna().all(), "Hay hospitales cuyo municipio no existe en dim_municipios"
dim_hos = dim_hos.rename(columns={"municipio": "municipio_hospital"})
dim_hos["etiqueta"] = dim_hos["hospital_id"] + " - " + dim_hos["institucion"]

# 4.2 Municipios, pacientes
dim_mun["tiene_hospital"] = dim_mun["municipio_id"].isin(dim_hos["municipio_id"]).astype(int)
dim_pac["grupo_edad_orden"] = dim_pac["grupo_edad"].map({"0-17": 1, "18-29": 2, "30-44": 3, "45-59": 4, "60+": 5})
dim_pac["tiene_atenciones"] = dim_pac["paciente_id"].isin(f["paciente_id"]).astype(int)

# 4.3 Triage (etiquetas descriptivas, sin metas normativas)
dim_tri = pd.DataFrame({
    "triage": [1, 2, 3, 4, 5],
    "nivel": ["1 - Critico", "2 - Emergencia", "3 - Urgente", "4 - Menos urgente", "5 - No urgente"],
    "grupo_prioridad": ["Alta (1-2)", "Alta (1-2)", "Media (3)", "Baja (4-5)", "Baja (4-5)"],
    "grupo_prioridad_orden": [1, 1, 2, 3, 3]})

# 4.4 Calendario completo (de dia 1 del primer mes a fin del ultimo mes)
meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
dsem = ["Lunes", "Martes", "Miercoles", "Jueves", "Viernes", "Sabado", "Domingo"]
dim_t = pd.DataFrame({"fecha": pd.date_range(f.fecha.min().replace(day=1), f.fecha.max() + pd.offsets.MonthEnd(0), freq="D")})
dim_t["anio"] = dim_t.fecha.dt.year
dim_t["trimestre"] = "T" + dim_t.fecha.dt.quarter.astype(str)
dim_t["anio_trimestre"] = dim_t.anio.astype(str) + "-" + dim_t.trimestre
dim_t["mes_num"] = dim_t.fecha.dt.month
dim_t["mes_nombre"] = dim_t.mes_num.map(lambda m: meses[m - 1])
dim_t["anio_mes"] = dim_t.fecha.dt.strftime("%Y-%m")
dim_t["semana_iso"] = dim_t.fecha.dt.isocalendar().week.astype(int)
dim_t["dia_semana_num"] = dim_t.fecha.dt.dayofweek + 1
dim_t["dia_semana"] = dim_t.dia_semana_num.map(lambda d: dsem[d - 1])
dim_t["es_fin_de_semana"] = (dim_t.dia_semana_num >= 6).astype(int)
chequeo("todas las fechas de la fact existen en dim_tiempo", f.fecha.isin(dim_t.fecha).all())

# ------------------------------------------------------------------ 5. EXPORTAR
print("Exportando archivos para Power BI...")
orden = ["atencion_id", "fecha_hora_ingreso", "fecha", "hora", "franja_horaria", "paciente_id", "hospital_id", "servicio_id",
         "diagnostico_id", "triage", "resultado_atencion", "tiempo_espera_min", "rango_espera", "rango_espera_orden",
         "estancia_dias", "costo_atencion_cop", "satisfaccion_1_a_5", "es_hospitalizado", "reingreso_30_dias",
         "espera_faltante", "costo_faltante", "satisfaccion_faltante", "espera_cero", "espera_mayor_120", "satisfaccion_baja",
         "remision_hospitalizada", "n_atencion_paciente", "dias_desde_atencion_previa", "reingreso_sin_antecedente_30d"]
f = f[orden].sort_values("atencion_id")
assert len(f) == len(x["fact_atenciones"]), "La fact cambio de numero de filas"

salidas = {"fact_atenciones": f, "dim_pacientes": dim_pac, "dim_hospitales": dim_hos, "dim_servicios": dim_ser,
           "dim_diagnosticos": dim_dia, "dim_municipios": dim_mun, "dim_aseguradoras": dim_ase,
           "dim_triage": dim_tri, "dim_tiempo": dim_t}

os.makedirs(CARPETA, exist_ok=True)
for nombre, df in salidas.items():
    df = df.copy()
    # Fechas como texto 'AAAA-MM-DD' (solo la fecha-hora lleva hora): asi 'fecha' de la fact y de dim_tiempo
    # tienen exactamente el mismo formato y la relacion en Power BI no falla.
    for col in df.select_dtypes(include=["datetime"]).columns:
        df[col] = df[col].dt.strftime("%Y-%m-%d %H:%M:%S" if col == "fecha_hora_ingreso" else "%Y-%m-%d")
    df.to_csv(f"{CARPETA}/{nombre}.csv", index=False, encoding="utf-8-sig")

bit = pd.DataFrame(bitacora)
bit.to_csv(f"{CARPETA}/bitacora_limpieza.csv", index=False, encoding="utf-8-sig")
print("\n--- BITACORA ---")
print(bit.to_string(index=False))
if (bit.resultado == "REVISAR").any():
    print("\n[!] Hay chequeos en REVISAR: mira la bitacora antes de seguir.")

with zipfile.ZipFile(ZIP_SALIDA, "w", zipfile.ZIP_DEFLATED) as zf:
    for archivo in sorted(os.listdir(CARPETA)):
        zf.write(os.path.join(CARPETA, archivo), archivo)

print(f"\nListo: {ZIP_SALIDA} ({len(salidas)} tablas + bitacora)")
if EN_COLAB:
    colab_files.download(ZIP_SALIDA)

Sube el archivo 'Hospital_360_dataset_6000reg.xlsx'


Saving Hospital_360_dataset_6000reg.xlsx to Hospital_360_dataset_6000reg (2).xlsx
Cargando datos...
Limpiando tabla de hechos...
Construyendo dimensiones...
Exportando archivos para Power BI...

--- BITACORA ---
                                                           chequeo resultado      detalle
                                 atencion_id unico (PK de la fact)        OK  0 repetidos
                                         sin filas 100% duplicadas        OK 0 duplicadas
                                    PK unica en dim de paciente_id        OK             
                                      FK paciente_id sin huerfanos        OK  0 huerfanos
                                    PK unica en dim de hospital_id        OK             
                                      FK hospital_id sin huerfanos        OK  0 huerfanos
                                    PK unica en dim de servicio_id        OK             
                                      FK servicio_id sin huerfanos  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>